In [7]:
import numpy as np
import torch
from sklearn.manifold import MDS
from scipy.spatial import KDTree, cKDTree
from joblib import Parallel, delayed
from scipy.spatial.distance import cdist
import umap
from umap.umap_ import fuzzy_simplicial_set, nearest_neighbors
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import faiss 
from sklearn.preprocessing import StandardScaler
from scipy.special import expit
from adbench.myutils import Utils
from adbench.run import RunPipeline
import pandas as pd
from numba import njit, prange
import math

# ---- Numba kernels (Euclidean Distance Version, cache=True) ----

@njit(parallel=True, fastmath=True, cache=True)
def _count_points_within_radius_from_D(D, eps):
    """
    Count neighbors within radius eps for each row of matrix D (Euclidean distance matrix).
    Excludes self (i==j).
    """
    n = D.shape[0]
    counts = np.zeros(n, np.int64)
    for i in prange(n):
        c = 0
        row = D[i]
        for j in range(n):
            if i == j:
                continue
            if row[j] <= eps:
                c += 1
        counts[i] = c
    return counts

@njit(fastmath=True, cache=True)
def _max_min_offdiag(D):
    """
    Compute maximum and minimum over off-diagonal entries of Euclidean distance matrix D.
    """
    n = D.shape[0]
    maxv = 0.0
    minv = 1e300
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            v = D[i, j]
            if v > maxv:
                maxv = v
            if v < minv:
                minv = v
    if minv == 1e300:
        minv = 0.0
    return maxv, minv


@njit(parallel=True, fastmath=True, cache=True)
def _pairwise_euclidean_distances_numba(X):
    """
    Compute pairwise Euclidean distances (n x n) for X (n, d).
    """
    n, d = X.shape
    D = np.empty((n, n), dtype=np.float64)
    for i in prange(n):
        xi = X[i]
        for j in range(n):
            s = 0.0
            xj = X[j]
            for k in range(d):
                tmp = xi[k] - xj[k]
                s += tmp * tmp
            D[i, j] = math.sqrt(s)
    return D

# ---- Updated functions using Euclidean Numba kernels ----

def count_points_within_radius(X, tree, epsilon):
    """
    Replacement for KDTree query_ball_tree using Euclidean numba distance matrix.
    """
    X_arr = np.array(X, dtype=np.float64)
    D = _pairwise_euclidean_distances_numba(X_arr)
    counts = _count_points_within_radius_from_D(D, epsilon)
    return counts

def find_k_nearest_neighbors(X, tree, query_point, k):
    if tree is not None:
        distances, indices = tree.query(query_point, k=k)
        return distances, indices
    else:
        X_arr = np.array(X)
        qp = np.atleast_2d(query_point)
        dists = cdist(qp, X_arr, metric='euclidean')
        k_eff = min(k, X_arr.shape[0])
        idx = np.argpartition(dists, k_eff-1, axis=1)[:, :k_eff]
        rows = np.arange(idx.shape[0])[:, None]
        sel_dists = dists[rows, idx]
        order = np.argsort(sel_dists, axis=1)
        idx_sorted = idx[rows, order]
        dists_sorted = sel_dists[rows, order]
        return dists_sorted, idx_sorted

def max_min_distances_kdtree(X):
    """
    Compute Euclidean max/min using numba kernel.
    """
    X_arr = np.array(X, dtype=np.float64)
    D = _pairwise_euclidean_distances_numba(X_arr)
    max_dist, min_dist = _max_min_offdiag(D)
    return max_dist, min_dist

def binary_search_condition(low, high, condition, tol=1e-4, max_iter=50):
    result = None
    for _ in range(max_iter):
        mid = (low + high) / 2
        if condition(mid):
            result = mid
            high = mid
        else:
            low = mid
        if abs(high - low) < tol:
            break
    return result

def condition_formulation(point_in_radius_counts, nbd_sample_count_threshold, satisfiability_proportion):
    satisfied = (point_in_radius_counts > nbd_sample_count_threshold).sum()
    return satisfied >= satisfiability_proportion


@njit(fastmath=True, cache=True)
def t_distribution_kernel_numba(X, nu):
    n, d = X.shape
    G = np.empty((n, n), np.float64)

    # Compute Gram matrix
    for i in range(n):
        for j in range(n):
            s = 0.0
            for k in range(d):
                s += X[i, k] * X[j, k]
            G[i, j] = s

    # Compute T-kernel from Gram matrix
    out = np.empty((n, n), np.float64)
    for i in range(n):
        for j in range(n):
            dist = G[i, i] + G[j, j] - 2.0 * G[i, j]
            out[i, j] = 1.0 / (1.0 + dist) ** nu + 1e-7
    return out



def get_empirical_weights(
    X,
    nbd_sample_count_threshold=5,
    max_iters_weight_count=4,
    satisfiability_proportion=0.3,
    n_neighbors=15,
    metric='euclidean',
    random_state=42,
    batch_size=1000 ):
    
    def umap_graph_similarity(X_batch):
        knn_indices, knn_dists, _ = nearest_neighbors(
            X_batch,
            n_neighbors=n_neighbors,
            metric=metric,
            metric_kwds={},
            angular=False,
            random_state=random_state,
            low_memory=True,
            use_pynndescent=True
        )

        G, _, _ = fuzzy_simplicial_set(
            X_batch,
            n_neighbors=n_neighbors,
            random_state=random_state,
            metric=metric,
            knn_indices=knn_indices,
            knn_dists=knn_dists,
            angular=False,
            set_op_mix_ratio=1.0,
            local_connectivity=1.0,
            apply_set_operations=True,
            verbose=False
        )
        return G.toarray() if isinstance(G, csr_matrix) else G

    def compute_weights_from_similarity(sim, X_batch):
        # NOTE: keeps original logic, but uses new numba-backed counting and max/min functions
        tree = None 

        # compute pairwise distances / extremes using numba helper
        max_dist, min_dist = max_min_distances_kdtree(sim)

        if nbd_sample_count_threshold >= len(X_batch):
            threshold = max(1, len(X_batch) - 1)
        else:
            threshold = nbd_sample_count_threshold

        effective_required = int(satisfiability_proportion * len(X_batch))

        # binary search using the numba-powered count_points_within_radius
        def cond(mid):
            counts = count_points_within_radius(sim, tree, mid)
            return condition_formulation(counts, threshold, effective_required)

        eps = binary_search_condition(
            min_dist, max_dist,
            cond
        )

        if eps is None:
            relaxed_thresh = max(1, threshold // 2)
            relaxed_prop = effective_required // 2

            def cond_relaxed(mid):
                counts = count_points_within_radius(sim, tree, mid)
                return condition_formulation(counts, relaxed_thresh, relaxed_prop)

            eps = binary_search_condition(
                min_dist, max_dist,
                cond_relaxed
            )

        if eps is None:
            eps = max_dist

        delta = (eps - 1e-6) / max_iters_weight_count
        all_counts = []

        for _ in range(max_iters_weight_count):
            counts = count_points_within_radius(sim, tree, eps)
            all_counts.append(counts)
            eps -= delta

        return np.mean(all_counts, axis=0)

    if len(X) <= 3 * n_neighbors:
        sim = umap_graph_similarity(X)
        return compute_weights_from_similarity(sim, X)

    effective_batch_size = min(batch_size, max(n_neighbors * 3, 100))
    total_batches = (len(X) + effective_batch_size - 1) // effective_batch_size

    weights_all = []
    for batch_idx in range(total_batches):
        start = batch_idx * effective_batch_size
        end = min(len(X), start + effective_batch_size)
        X_batch = X[start:end]

        sim = umap_graph_similarity(X_batch)
        weights_batch = compute_weights_from_similarity(sim, X_batch)
        weights_all.append(weights_batch)

    return np.concatenate(weights_all)

def shift_data_torch(X, indices, weights, learning_rate):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    X_t = torch.from_numpy(X).float().to(device)
    indices_t = torch.from_numpy(indices).long().to(device)
    weights_t = torch.from_numpy(weights).float().to(device)

    data_nebs = X_t[indices_t]
    weights_nebs = weights_t[indices_t]
    weights_nebs = weights_nebs / (weights_nebs.sum(dim=1, keepdim=True) + 1e-6)

    new_d = torch.sum(weights_nebs.unsqueeze(2) * data_nebs, dim=1)
    change = torch.norm(X_t - new_d, dim=1)
    unit_vec = (new_d - X_t) / (change.unsqueeze(1) + 1e-6)

    revised_d = X_t + learning_rate * change.unsqueeze(1) * unit_vec

    return revised_d.cpu().numpy(), change.cpu().numpy()

def get_shift_fast(X, k, nbd_sample_count_threshold, learning_rate, max_iters_shift, shift_threshold, return_weights=False):

    weights = get_empirical_weights(
        X,
        nbd_sample_count_threshold=nbd_sample_count_threshold,
        max_iters_weight_count=4,
        satisfiability_proportion=0.3,
        batch_size=1000
    )

    n_samples = len(X)
    shifted_dataset = X.copy()
    total_distance = np.zeros(n_samples)

    index = faiss.IndexFlatL2(X.shape[1])
    index.add(X.astype(np.float32))

    for _ in range(max_iters_shift):
        changes = []
        start, end = 0, n_samples

        d = shifted_dataset[start:end]
        _, indices = index.search(d.astype(np.float32), k)

        revised_d, change = shift_data_torch(shifted_dataset, indices, weights, learning_rate)

        total_distance[start:end] += change
        shifted_dataset[start:end] = revised_d
        changes.extend(change.tolist())

        if np.mean(changes) < shift_threshold:
            break

    if return_weights:
        return shifted_dataset, weights, total_distance
    else:
        return shifted_dataset, total_distance

def mean_shift_manifold_learning(X, k=30, nbd_sample_count_threshold=30, learning_rate=.3,
                                 max_iters_shift=10, shift_threshold=0.0001, return_weights=False):

    if not return_weights:
        data_shifted, total_distance = get_shift_fast(
            X, k, nbd_sample_count_threshold, learning_rate, max_iters_shift, shift_threshold
        )
        return data_shifted, total_distance

    else:
        data_shifted, weights, total_distance = get_shift_fast(
            X, k, nbd_sample_count_threshold, learning_rate, max_iters_shift, shift_threshold, True
        )
        return data_shifted, weights, total_distance

class MSML:
    def __init__(self, seed: int, model_name: str = 'MSML', k=12,
                 nbd_sample_count_threshold=70, learning_rate=.1,
                 max_iters_shift=6, shift_threshold=0.003,
                 anomalyThreshold=0.22, scaler=StandardScaler()):
        self.k = k
        self.nbd_sample_count_threshold = nbd_sample_count_threshold
        self.learning_rate = learning_rate
        self.max_iters_shift = max_iters_shift
        self.shift_threshold = shift_threshold
        self.anomalyThreshold = anomalyThreshold
        self.scaler = scaler
        self.seed = seed
        self.utils = Utils()
        self.model_name = model_name

    def fit(self, X_train, y_train=None):
        return self

    def predict_score(self, X):
        data_shifted, total_distance = mean_shift_manifold_learning(
            X,
            self.k,
            self.nbd_sample_count_threshold,
            self.learning_rate,
            self.max_iters_shift,
            self.shift_threshold
        )

        total_distance = self.scaler.fit_transform(total_distance.reshape(-1, 1))
        total_distance = expit(total_distance)

        return total_distance.squeeze()

def warm_up_numba():
    """
    Runs the MSML pipeline on a small dummy dataset to trigger Numba JIT compilation.
    This ensures the compilation time doesn't count towards the benchmark.
    """
    print("--- Warming up Numba Kernels ---")
    # Create dummy data of type float64 (matching what we use in real execution)
    dummy_X = np.random.rand(50, 10).astype(np.float64)
    
    # Run the main logic with minimal parameters to be fast
    # This touches: get_empirical_weights -> binary search -> numba distance kernels
    mean_shift_manifold_learning(
        dummy_X, 
        k=5, 
        nbd_sample_count_threshold=5, 
        learning_rate=0.1, 
        max_iters_shift=1,  # One iteration is enough to compile
        shift_threshold=0.1
    )
    print("--- Warm-up Complete ---")

def convert_numpy_inplace(obj):
    if isinstance(obj, (np.integer, np.floating)):
        return obj.item()
    if isinstance(obj, dict):
        return {k: convert_numpy_inplace(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return type(obj)(convert_numpy_inplace(v) for v in obj)
    return obj



import builtins

_original_print = builtins.print

def patched_print(*args, **kwargs):
    cleaned_args = [convert_numpy_inplace(a) for a in args]
    return _original_print(*cleaned_args, **kwargs)

builtins.print = patched_print



if __name__ == "__main__":
    utils = Utils()
    utils.download_datasets()

    # 1. Execute Warm-up
    warm_up_numba()

    # 2. Start Benchmark
    pipeline = RunPipeline(
        suffix='ADBench',
        parallel='unsupervise',
        realistic_synthetic_mode='dependency',
        noise_type='irrelevant_features'
    )

    results = pipeline.run(clf=MSML)
    pd.DataFrame(results).to_csv('adbench/result/MSML4.csv', index=False)

    results2 = pipeline.run()
    pd.DataFrame(results2).to_csv('adbench/result/benchmarks4.csv', index=False)



if there is any question while downloading datasets, we suggest you to download it from the website:
https://github.com/Minqi824/ADBench/tree/main/adbench/datasets
如果您在中国大陆地区，请使用链接：
https://jihulab.com/BraudoCC/ADBench_datasets/
100% [................................................................................] 3852 / 3852

100%|██████████████████████████████████████████████████████████████████████████████████| 3/3 [00:00<00:00, 1075.74it/s]

CIFAR10_0.npz already exists. Skipping download...
CIFAR10_1.npz already exists. Skipping download...
CIFAR10_2.npz already exists. Skipping download...
CIFAR10_3.npz already exists. Skipping download...
CIFAR10_4.npz already exists. Skipping download...
CIFAR10_5.npz already exists. Skipping download...
CIFAR10_6.npz already exists. Skipping download...
CIFAR10_7.npz already exists. Skipping download...
CIFAR10_8.npz already exists. Skipping download...
CIFAR10_9.npz already exists. Skipping download...
FashionMNIST_0.npz already exists. Skipping download...
FashionMNIST_1.npz already exists. Skipping download...
FashionMNIST_2.npz already exists. Skipping download...
FashionMNIST_3.npz already exists. Skipping download...
FashionMNIST_4.npz already exists. Skipping download...
FashionMNIST_5.npz already exists. Skipping download...
FashionMNIST_6.npz already exists. Skipping download...
FashionMNIST_7.npz already exists. Skipping download...
FashionMNIST_8.npz already exists. Skippin

--- Warm-up Complete ---
subsampling for dataset 10_cover...
current noise type: None
{'Samples': 10000, 'Features': 10, 'Anomalies': 97, 'Anomalies Ratio(%)': 0.97}
subsampling for dataset 10_cover...
current noise type: None
{'Samples': 10000, 'Features': 10, 'Anomalies': 116, 'Anomalies Ratio(%)': 1.16}
subsampling for dataset 10_cover...
current noise type: None
{'Samples': 10000, 'Features': 10, 'Anomalies': 95, 'Anomalies Ratio(%)': 0.95}
subsampling for dataset 11_donors...
current noise type: None
{'Samples': 10000, 'Features': 10, 'Anomalies': 618, 'Anomalies Ratio(%)': 6.18}
subsampling for dataset 11_donors...
current noise type: None
{'Samples': 10000, 'Features': 10, 'Anomalies': 578, 'Anomalies Ratio(%)': 5.78}
subsampling for dataset 11_donors...
current noise type: None
{'Samples': 10000, 'Features': 10, 'Anomalies': 597, 'Anomalies Ratio(%)': 5.97}
current noise type: None
{'Samples': 1941, 'Features': 27, 'Anomalies': 673, 'Anomalies Ratio(%)': 34.67}
current noise ty

KeyboardInterrupt: 